# Notebook 1 — Adquisición de Datos e Ingestión

**Objetivo**: descargar de forma 100% programática y reproducible las dos fuentes de datos
del sistema, sin depender de ningún fichero pre-descargado:

1. **Histórico de partidos internacionales** — espejo público en GitHub del dataset de Kaggle
   `martj42/international-football-results` (no requiere cuenta ni API key).
2. **Histórico de ELO por selección y fecha** — `eloratings.net`, que publica por cada
   selección un fichero TSV con su Elo partido a partido desde 1900 en adelante (tampoco
   requiere cuenta).

Ninguna de las dos fuentes se scrapea desde un sitio que lo prohíba: la primera es un
repo público pensado para reutilizarse, y la segunda no impone ningún término de servicio
restrictivo (se comprobó su `robots.txt`, inexistente, y no hay límite de tasa documentado).
Aun así, se respeta una pausa entre peticiones al histórico de ELO por cortesía, al ser un
sitio mantenido por una sola persona.

**Salida**: `data/raw/results.csv` (histórico completo) y `data/raw/elo_historico.csv`
(Elo diario de las selecciones que han jugado el Mundial 2026).

In [1]:
import io
import json
import re
import time
import unicodedata
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

import sys

# El kernel de Jupyter usa como directorio de trabajo la carpeta del propio
# notebook (notebooks/); las rutas de datos son relativas a la RAÍZ del
# proyecto (un nivel por encima), sea cual sea la herramienta con la que se
# ejecute (JupyterLab, VS Code, `jupyter nbconvert --execute`).
DIR_RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_RAW = DIR_RAIZ / "data" / "raw"
DIR_RAW.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

## 1.1 Histórico de partidos internacionales

`martj42/international_results` es el repositorio de origen del dataset de Kaggle
"International football results from 1872 to today": el mismo autor lo mantiene también
en GitHub, actualizado, y sirve el CSV en bruto vía `raw.githubusercontent.com` — no hace
falta ni cuenta ni token para leerlo con una petición HTTP normal.

In [2]:
URL_HISTORICO = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"
# Companion del mismo dataset: desenlaces de tandas de penaltis (ganador y primer
# lanzador) -- lo usa el Notebook 2 (2.6b) para atribuir títulos decididos en tanda.
URL_SHOOTOUTS = "https://raw.githubusercontent.com/martj42/international_results/master/shootouts.csv"


def descargar_historico_partidos(url: str = URL_HISTORICO, timeout: int = 30) -> pd.DataFrame:
    """Descarga el CSV de resultados internacionales directamente desde GitHub.

    No se usa la API de Kaggle a propósito: exigiría credenciales (`kaggle.json`)
    solo para leer un fichero que el propio autor del dataset ya publica en
    abierto en su repositorio — una petición HTTP simple es la vía más
    reproducible (cualquiera puede ejecutar esta celda sin darse de alta en nada).

    Devuelve el DataFrame crudo, tal cual llega, sin limpiar todavía: la limpieza
    y el tipado se hacen explícitamente en la validación (más abajo) para que
    quede claro qué se corrige y por qué.
    """
    respuesta = requests.get(url, timeout=timeout)
    respuesta.raise_for_status()
    df = pd.read_csv(io.StringIO(respuesta.text))
    print(f"Descargados {len(df):,} partidos desde {url}")
    return df


df_historico_crudo = descargar_historico_partidos()
df_historico_crudo.tail()

Descargados 49,505 partidos desde https://raw.githubusercontent.com/martj42/international_results/master/results.csv


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49500,2026-07-07,Switzerland,Colombia,0.0,0.0,FIFA World Cup,Vancouver,Canada,True
49501,2026-07-09,France,Morocco,NaN,NaN,FIFA World Cup,Foxborough,United States,True
49502,2026-07-10,Spain,Belgium,NaN,NaN,FIFA World Cup,Inglewood,United States,True
49503,2026-07-11,Norway,England,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True
49504,2026-07-11,Argentina,Switzerland,NaN,NaN,FIFA World Cup,Kansas City,United States,True


## 1.2 Histórico de ELO por selección

`eloratings.net` no expone una API formal, pero cada selección tiene una página propia
con su historial completo servido como TSV plano (`https://www.eloratings.net/<Selección>.tsv`,
con espacios sustituidos por guion bajo). Cada fila es un partido de esa selección, con el
Elo **posterior** al partido para ambos lados.

Para no tener que adivinar ni mantener a mano la lista de selecciones clasificadas al
Mundial 2026, se extraen directamente del propio histórico de partidos: cualquier
selección que aparezca jugando el torneo `FIFA World Cup` en 2026 es, por definición,
una de las 48 clasificadas — así el notebook no depende de ninguna lista fija.

In [3]:
def extraer_selecciones_mundial_2026(df_historico: pd.DataFrame) -> list[str]:
    """Selecciones que han jugado el Mundial 2026, derivadas del propio histórico.

    Se evita hardcodear la lista de clasificados: si el torneo cambia de
    formato o el dataset se actualiza, esta función sigue siendo correcta
    sin tocar el notebook.
    """
    fechas = pd.to_datetime(df_historico["date"], errors="coerce")
    es_mundial_2026 = (df_historico["tournament"] == "FIFA World Cup") & (fechas.dt.year == 2026)
    partidos_mundial = df_historico[es_mundial_2026]
    equipos = sorted(set(partidos_mundial["home_team"]) | set(partidos_mundial["away_team"]))
    print(f"Selecciones detectadas en el Mundial 2026: {len(equipos)}")
    return equipos


SELECCIONES_MUNDIAL_2026 = extraer_selecciones_mundial_2026(df_historico_crudo)
SELECCIONES_MUNDIAL_2026[:10]

Selecciones detectadas en el Mundial 2026: 48


['Algeria',
 'Argentina',
 'Australia',
 'Austria',
 'Belgium',
 'Bosnia and Herzegovina',
 'Brazil',
 'Canada',
 'Cape Verde',
 'Colombia']

In [4]:
# Un puñado de selecciones no se llaman igual en eloratings.net que en el
# histórico de partidos (acentos, apóstrofes, nombres oficiales distintos). Se
# resuelve intentando primero el nombre tal cual y, si falla, una lista corta
# de alias conocidos — no hay forma de saberlo sin probar contra el sitio real.
ALIAS_ELORATINGS = {
    "Curaçao": "Curacao",
    "Czech Republic": "Czechia",
    "Ivory Coast": "Ivory_Coast",
    "United States": "United_States",
    "South Korea": "South_Korea",
    "DR Congo": "DR_Congo",
    "Cape Verde": "Cape_Verde",
    "New Zealand": "New_Zealand",
    "South Africa": "South_Africa",
    "Saudi Arabia": "Saudi_Arabia",
    # "Bosnia and Herzegovina" NO necesita alias: el reemplazo por defecto
    # (espacios -> guion bajo) ya produce el slug correcto ("Bosnia_and_
    # Herzegovina"), verificado contra el sitio real.
}


def _slug_eloratings(equipo: str) -> str:
    """Convierte el nombre de una selección al slug de URL de eloratings.net."""
    if equipo in ALIAS_ELORATINGS:
        return ALIAS_ELORATINGS[equipo]
    return equipo.replace(" ", "_")


def descargar_elo_equipo(equipo: str, sesion: requests.Session, timeout: int = 10) -> pd.DataFrame | None:
    """Descarga y parsea el TSV histórico de Elo de una selección.

    Formato de cada fila (sin cabecera): año, mes, día, código_local,
    código_visitante, goles_local, goles_visitante, competición, (vacío),
    peso_del_partido, elo_local_después, elo_visitante_después, ...
    El código de 2-3 letras que usa eloratings.net no es un estándar (ISO ni
    FIFA) documentado públicamente, así que en vez de mantener una tabla de
    equivalencias se infiere: el código de ESTA selección es el que más se
    repite entre local y visitante a lo largo de todo su historial (un rival
    puntual aparece muchas menos veces).
    """
    url = f"https://www.eloratings.net/{_slug_eloratings(equipo)}.tsv"
    try:
        resp = sesion.get(url, timeout=timeout, headers={"User-Agent": "Mozilla/5.0 (research use)"})
        if resp.status_code != 200 or not resp.text.strip():
            print(f"  [aviso] sin histórico de Elo para {equipo!r} (HTTP {resp.status_code})")
            return None
    except requests.RequestException as exc:
        print(f"  [aviso] fallo de red descargando Elo de {equipo!r}: {exc}")
        return None

    columnas = ["anio", "mes", "dia", "cod_local", "cod_visitante", "goles_local", "goles_visitante",
                "competicion", "_vacio", "peso_partido", "elo_local_despues", "elo_visitante_despues",
                "_c1", "_c2", "_c3", "_c4"]
    df = pd.read_csv(io.StringIO(resp.text), sep="\t", header=None, names=columnas)
    df["fecha"] = pd.to_datetime(dict(year=df.anio, month=df.mes, day=df.dia), errors="coerce")

    codigo_propio = pd.concat([df["cod_local"], df["cod_visitante"]]).value_counts().idxmax()
    df["elo"] = df["elo_local_despues"].where(df["cod_local"] == codigo_propio, df["elo_visitante_despues"])
    df["equipo"] = equipo
    return df.dropna(subset=["fecha", "elo"])[["fecha", "equipo", "elo"]]


def descargar_elo_historico(selecciones: list[str], pausa: float = 0.4) -> pd.DataFrame:
    """Descarga el Elo histórico de cada selección de la lista y lo concatena.

    La pausa entre peticiones no responde a un límite de tasa documentado
    (no lo hay), es cortesía hacia un sitio mantenido por una sola persona.
    """
    sesion = requests.Session()
    piezas = []
    for i, equipo in enumerate(selecciones, start=1):
        df_equipo = descargar_elo_equipo(equipo, sesion)
        if df_equipo is not None:
            piezas.append(df_equipo)
        if i % 10 == 0:
            print(f"  ... {i}/{len(selecciones)} selecciones procesadas")
        time.sleep(pausa)

    if not piezas:
        raise RuntimeError("No se pudo descargar ningún histórico de Elo.")
    df_elo = pd.concat(piezas, ignore_index=True).sort_values(["equipo", "fecha"]).reset_index(drop=True)
    print(f"\nElo descargado: {len(df_elo):,} filas para {df_elo['equipo'].nunique()}/{len(selecciones)} selecciones")
    return df_elo


df_elo_crudo = descargar_elo_historico(SELECCIONES_MUNDIAL_2026)
df_elo_crudo.tail()

  ... 10/48 selecciones procesadas


  ... 20/48 selecciones procesadas


  [aviso] fallo de red descargando Elo de 'Netherlands': HTTPSConnectionPool(host='www.eloratings.net', port=443): Read timed out. (read timeout=10)


  [aviso] fallo de red descargando Elo de 'New Zealand': HTTPSConnectionPool(host='www.eloratings.net', port=443): Read timed out. (read timeout=10)
  ... 30/48 selecciones procesadas


  ... 40/48 selecciones procesadas



Elo descargado: 34,767 filas para 46/48 selecciones


,fecha,equipo,elo
34762,2026-06-01,Uzbekistan,1718
34763,2026-06-08,Uzbekistan,1714
34764,2026-06-17,Uzbekistan,1698
34765,2026-06-23,Uzbekistan,1677
34766,2026-06-27,Uzbekistan,1631


## 1.3 Validación de los datos entrantes

Antes de persistir nada: tipos correctos, sin nulos en las columnas clave, fechas
parseables, y rangos físicamente posibles (no goles negativos, no fechas futuras
absurdas). Fallar aquí con un mensaje claro es mucho más barato que descubrir el
problema tres notebooks más tarde, con el error ya diluido entre features.

In [5]:
def validar_historico_partidos(df: pd.DataFrame) -> pd.DataFrame:
    """Valida y tipa el histórico de partidos. Lanza AssertionError con un
    mensaje claro si algo no cumple el contrato esperado; nunca corrige en
    silencio un dato que debería investigarse.
    """
    columnas_esperadas = {"date", "home_team", "away_team", "home_score", "away_score", "tournament", "neutral"}
    faltantes = columnas_esperadas - set(df.columns)
    assert not faltantes, f"Faltan columnas esperadas en el histórico: {faltantes}"

    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    filas_sin_fecha = df["date"].isna().sum()
    assert filas_sin_fecha < len(df) * 0.01, f"Demasiadas fechas inválidas: {filas_sin_fecha}"
    df = df.dropna(subset=["date"])

    assert df["date"].max() <= pd.Timestamp.today() + pd.Timedelta(days=180), "Hay fechas demasiado en el futuro"

    for col in ("home_team", "away_team"):
        assert df[col].notna().all(), f"Nulos en {col}"
        df[col] = df[col].str.strip()

    # Los goles pueden ser NaN (partidos programados que aún no se han jugado,
    # p.ej. del propio Mundial 2026 en curso) — eso es válido, no un error; lo
    # que no es válido es un marcador negativo en los que SÍ tienen resultado.
    for col in ("home_score", "away_score"):
        con_resultado = df[col].notna()
        assert (df.loc[con_resultado, col] >= 0).all(), f"Goles negativos encontrados en {col}"

    print(f"Validación OK: {len(df):,} partidos, {filas_sin_fecha} filas descartadas por fecha inválida")
    return df


def validar_elo_historico(df: pd.DataFrame) -> pd.DataFrame:
    """Mismo criterio que arriba, para el histórico de Elo."""
    df = df.copy()
    assert df["equipo"].notna().all(), "Nulos en 'equipo'"
    assert df["fecha"].notna().all(), "Nulos en 'fecha'"
    assert df["elo"].between(0, 4000).all(), "Valores de Elo fuera de un rango físicamente razonable"
    print(f"Validación OK: {len(df):,} filas de Elo, {df['equipo'].nunique()} selecciones")
    return df


df_historico = validar_historico_partidos(df_historico_crudo)
df_elo = validar_elo_historico(df_elo_crudo)

Validación OK: 49,505 partidos, 0 filas descartadas por fecha inválida
Validación OK: 34,767 filas de Elo, 46 selecciones


## 1.4 Persistencia en `data/raw/`

In [6]:
ruta_shootouts = DIR_RAW / "shootouts.csv"
pd.read_csv(URL_SHOOTOUTS).to_csv(ruta_shootouts, index=False)
print(f"Guardado: {ruta_shootouts}")

ruta_historico = DIR_RAW / "results.csv"
ruta_elo = DIR_RAW / "elo_historico.csv"

df_historico.to_csv(ruta_historico, index=False)
df_elo.to_csv(ruta_elo, index=False)

print(f"Guardado: {ruta_historico} ({len(df_historico):,} filas)")
print(f"Guardado: {ruta_elo} ({len(df_elo):,} filas)")

Guardado: /Users/danielcanteragomez/portfolio/wc-2026-match-predictor/data/raw/shootouts.csv
Guardado: /Users/danielcanteragomez/portfolio/wc-2026-match-predictor/data/raw/results.csv (49,505 filas)
Guardado: /Users/danielcanteragomez/portfolio/wc-2026-match-predictor/data/raw/elo_historico.csv (34,767 filas)


## 1.5 Calendario del Mundial 2026: resultado exacto, prórroga y penaltis incluidos

El espejo de GitHub (sección 1.1) es fiable pero solo guarda el marcador de **los 90
minutos**, nunca quién avanza tras prórroga o penaltis -- y durante el propio Mundial
puede tardar horas o días en reflejar un partido recién acabado. Para cubrir ambos
huecos, esta sección descarga automáticamente
[`openfootball/worldcup.json`](https://github.com/openfootball/worldcup.json)
(`2026/worldcup.json`), un calendario/resultados del torneo mantenido en abierto que sí
incluye prórroga (`et`) y penaltis (`p`) cuando aplican, y lo fusiona con
`data/raw/wc2026_calendario.json`.

La fusión, no sobrescritura ciega, importa: si un partido recién jugado todavía no
aparece en la fuente descargada (puede tardar), pero SÍ está anotado a mano en el
fichero local (`score.ft`, y `et`/`p` si se decidió en prórroga/penaltis), ese dato local
se conserva -- la fuente descargada solo manda cuando ella misma ya trae marcador para
ese partido. Así, editar `data/raw/wc2026_calendario.json` a mano para adelantarse a la
fuente pública sigue funcionando aunque se vuelva a ejecutar este notebook.

Si la descarga falla (sin red), se usa la copia local existente si la hay; si no existe
ninguna, esta sección simplemente no aporta nada y el resto del pipeline sigue
funcionando solo con el mirror de la 1.1. Solo se leen, de ese fichero, los partidos que
**ya tienen marcador de 90 minutos** (`score.ft`) -- un partido programado, o un cruce de
eliminatoria que todavía depende de un resultado anterior, no trae esa clave, y los
placeholders de bracket (`"W83"`, `"L101"`) los resuelve la propia fuente en cuanto se
conoce el equipo real, así que no hace falta interpretarlos aquí.


In [7]:
URL_CALENDARIO_MUNDIAL = "https://raw.githubusercontent.com/openfootball/worldcup.json/master/2026/worldcup.json"
RUTA_CALENDARIO_JSON = DIR_RAW / "wc2026_calendario.json"
PAISES_SEDE_MUNDIAL_2026 = {"Mexico", "Canada", "United States"}

# Un puñado de selecciones no se llaman igual en el JSON que en el histórico
# de partidos (mismo motivo que ALIAS_ELORATINGS más arriba).
MAPEO_NOMBRES_JSON = {
    "Bosnia & Herzegovina": "Bosnia and Herzegovina",
    "USA": "United States",
}

# Las 16 sedes del Mundial 2026, para reconstruir el país de cada partido; el
# JSON mete la ciudad-sede real entre paréntesis cuando difiere del área
# metropolitana (p.ej. "Guadalajara (Zapopan)" -> ciudad "Zapopan").
CIUDAD_A_PAIS_MUNDIAL_2026 = {
    "Mexico City": "Mexico", "Zapopan": "Mexico", "Guadalupe": "Mexico",
    "Toronto": "Canada", "Vancouver": "Canada",
    "Inglewood": "United States", "Santa Clara": "United States",
    "East Rutherford": "United States", "Foxborough": "United States",
    "Philadelphia": "United States", "Houston": "United States",
    "Seattle": "United States", "Atlanta": "United States",
    "Miami Gardens": "United States", "Arlington": "United States",
    "Kansas City": "United States",
}


def _ciudad_desde_sede(sede: str) -> str:
    return sede.split("(")[1].rstrip(")") if "(" in sede else sede


def _determinar_local_visitante(equipo1: str, equipo2: str, pais_sede: str) -> tuple[str, str]:
    """El orden `team1`/`team2` del JSON NO es fiable como local/visitante:
    en la última jornada de grupos (kickoff simultáneo) se ha comprobado que
    a veces lista primero al NO anfitrión aunque el partido se juegue en el
    país de la otra selección (p.ej. "team1": "Czech Republic", "team2":
    "Mexico" para un México-Chequia jugado en Ciudad de México). La sede es
    la señal fiable: si una de las dos selecciones coincide con el país
    anfitrión de esa sede, esa es la local, gane o pierda. Si ninguna de las
    dos es el país anfitrión (partido neutral de verdad), se respeta el
    orden del JSON.
    """
    if equipo1 == pais_sede:
        return equipo1, equipo2
    if equipo2 == pais_sede:
        return equipo2, equipo1
    return equipo1, equipo2


def descargar_calendario_json_mundial(url: str | None = None, timeout: int = 30) -> dict:
    """Descarga el calendario/resultados del Mundial 2026 desde `openfootball/worldcup.json`.

    A diferencia del mirror de la sección 1.1 (que solo guarda el marcador de
    los 90 minutos), esta fuente incluye prórroga (`et`) y penaltis (`p`)
    cuando aplican -- imprescindible para saber quién avanza de verdad en un
    cruce de eliminatoria que empató a 90 minutos.

    `url` por defecto usa `URL_CALENDARIO_MUNDIAL` (no como valor por defecto
    del parámetro: los tests extraen esta función de forma aislada vía `ast`,
    sin la constante de nivel superior, y un default que la referenciara
    fallaría con `NameError` al definirse).

    El fichero de origen no numera los partidos, pero los notebooks 4 y 5
    necesitan un `num` (73-104, dieciseisavos a final) para identificar
    partidos de eliminatoria y resolver referencias de bracket ("W83",
    "L101") -- solo se asigna a esos 32 partidos, nunca a los de fase de
    grupos (1-72), porque ambos notebooks asumen que la sola PRESENCIA de
    `num` en un partido ya significa "es de eliminatoria". Verificado contra
    el calendario oficial: la posición 1-indexada dentro de `matches`
    coincide con el número de partido FIFA para las 104 fechas del torneo.
    """
    url = url or URL_CALENDARIO_MUNDIAL
    respuesta = requests.get(url, timeout=timeout)
    respuesta.raise_for_status()
    calendario = respuesta.json()
    for i, partido in enumerate(calendario["matches"], start=1):
        if i >= 73:
            partido["num"] = i
    return calendario


def _fusionar_calendario(calendario_local: dict | None, calendario_descargado: dict) -> dict:
    """Combina lo recién descargado con lo que ya hubiera en local: la fuente
    descargada manda siempre que YA tenga marcador para un partido, pero si un
    partido solo tiene marcador en el local (alguien lo anotó a mano porque la
    fuente pública todavía no lo refleja) ese marcador se conserva -- sin esto,
    cada ejecución de esta celda borraría cualquier resultado metido a mano
    que la fuente pública aún no haya recogido.
    """
    if calendario_local is None:
        return calendario_descargado
    marcadores_locales = {
        (p["date"], p["team1"], p["team2"]): p.get("score") for p in calendario_local["matches"]
    }
    for partido in calendario_descargado["matches"]:
        clave = (partido["date"], partido["team1"], partido["team2"])
        if partido.get("score") is None and marcadores_locales.get(clave) is not None:
            partido["score"] = marcadores_locales[clave]
    return calendario_descargado


try:
    calendario_mundial = descargar_calendario_json_mundial()
    calendario_local_previo = None
    if RUTA_CALENDARIO_JSON.exists():
        with open(RUTA_CALENDARIO_JSON, encoding="utf-8") as f:
            calendario_local_previo = json.load(f)
    calendario_mundial = _fusionar_calendario(calendario_local_previo, calendario_mundial)
    with open(RUTA_CALENDARIO_JSON, "w", encoding="utf-8") as f:
        json.dump(calendario_mundial, f, ensure_ascii=False, indent=1)
    print(f"Calendario descargado: {len(calendario_mundial['matches'])} partidos -> {RUTA_CALENDARIO_JSON}")
except requests.RequestException as exc:
    if RUTA_CALENDARIO_JSON.exists():
        print(f"[aviso] fallo de red descargando el calendario ({exc}); se usa la copia local existente.")
    else:
        print(f"[aviso] fallo de red descargando el calendario ({exc}) y no hay copia local -- se omite esta sección.")


def parsear_calendario_json_mundial(ruta_json: Path) -> pd.DataFrame:
    """Convierte el calendario/resultados del Mundial 2026 en JSON al esquema
    de `results.csv`. Solo produce filas para partidos que ya tienen marcador
    de 90 minutos (`score.ft`); un partido sin jugar, o un cruce de
    eliminatoria pendiente de un resultado anterior, se ignora.
    """
    with open(ruta_json, encoding="utf-8") as f:
        calendario = json.load(f)

    filas = []
    for partido in calendario["matches"]:
        marcador = partido.get("score")
        if marcador is None or "ft" not in marcador:
            continue

        equipo1 = MAPEO_NOMBRES_JSON.get(partido["team1"], partido["team1"])
        equipo2 = MAPEO_NOMBRES_JSON.get(partido["team2"], partido["team2"])
        ciudad = _ciudad_desde_sede(partido["ground"])
        pais = CIUDAD_A_PAIS_MUNDIAL_2026[ciudad]
        equipo_local, equipo_visitante = _determinar_local_visitante(equipo1, equipo2, pais)

        goles1, goles2 = marcador["ft"]
        goles_local, goles_visitante = (goles1, goles2) if equipo_local == equipo1 else (goles2, goles1)

        filas.append({
            "date": partido["date"],
            "home_team": equipo_local,
            "away_team": equipo_visitante,
            "home_score": goles_local,
            "away_score": goles_visitante,
            "tournament": "FIFA World Cup",
            "city": ciudad,
            "country": pais,
            "neutral": not (equipo_local in PAISES_SEDE_MUNDIAL_2026 and equipo_local == pais),
        })

    df_nuevo = pd.DataFrame(filas)
    df_nuevo["date"] = pd.to_datetime(df_nuevo["date"])
    return df_nuevo


if RUTA_CALENDARIO_JSON.exists():
    df_json = parsear_calendario_json_mundial(RUTA_CALENDARIO_JSON)

    # Idempotente: un partido ya presente (misma fecha + equipos) se
    # sobrescribe con el marcador del JSON en vez de duplicarse.
    clave = ["date", "home_team", "away_team"]
    ya_presentes = df_historico.merge(df_json[clave], on=clave, how="inner")
    df_historico_sin_solapar = df_historico.merge(df_json[clave], on=clave, how="left", indicator=True)
    df_historico_sin_solapar = df_historico_sin_solapar[df_historico_sin_solapar["_merge"] == "left_only"].drop(columns="_merge")

    df_historico = pd.concat([df_historico_sin_solapar, df_json], ignore_index=True).sort_values("date").reset_index(drop=True)
    df_historico = validar_historico_partidos(df_historico)
    df_historico.to_csv(ruta_historico, index=False)

    print(f"Calendario JSON: {len(df_json)} partidos con marcador ({len(ya_presentes)} ya estaban en el mirror y se "
          f"han refrescado con el JSON, {len(df_json) - len(ya_presentes)} son nuevos).")
    print(f"{ruta_historico} reescrito: {len(df_historico):,} partidos totales.")
else:
    print(f"No hay calendario JSON en {RUTA_CALENDARIO_JSON} — se usa solo el mirror de GitHub.")


Calendario descargado: 104 partidos -> /Users/danielcanteragomez/portfolio/wc-2026-match-predictor/data/raw/wc2026_calendario.json
Validación OK: 49,505 partidos, 0 filas descartadas por fecha inválida
Calendario JSON: 96 partidos con marcador (96 ya estaban en el mirror y se han refrescado con el JSON, 0 son nuevos).
/Users/danielcanteragomez/portfolio/wc-2026-match-predictor/data/raw/results.csv reescrito: 49,505 partidos totales.


## Resumen

| Fichero | Filas | Cobertura |
|---|---|---|
| `data/raw/results.csv` | histórico completo internacional | 1872 → hoy |
| `data/raw/elo_historico.csv` | 48 selecciones del Mundial 2026 | Elo partido a partido |
| `data/raw/wc2026_calendario.json` | 104 partidos del Mundial 2026 | calendario completo, con prórroga/penaltis |

Siguiente paso: **Notebook 2**, que cruza ambas fuentes y construye las features con
ventanas móviles, sin fuga de información temporal.

La sección 1.5 ya ha descargado el calendario del Mundial 2026 y refrescado
`results.csv` con sus partidos (marcador de 90 minutos) antes de este resumen — sin
necesitar ningún fichero mantenido a mano.
